# BEE 4750 Homework 2: Systems Modeling and Simulation

**Name**: Regina Velasco and Parker Catena

**ID**: rv297 and pdc76

> **Due Date**
>
> Thursday, 09/25/25, 9:00pm

## Overview

### Instructions

-   Problem 1 asks you to draw a systems diagram and identify the type
    of a feedback.
-   Problem 2 asks you to model contaminant concentrations in a river
    and use simulation to compare the concentrations to a regulatory
    standard.
-   Problem 3 asks you to explore the implications of an ice-albedo
    feedback in the Earth’s climate system by understanding the
    equilibria of the modeled system and their stabilities.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [5]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/BEE 4750 /HW 2/hw2-regina`


In [6]:
using Plots
using LaTeXStrings
using CSV
using DataFrames
using Roots

## Problems (Total: 30 Points)

### Problem 1 (5 points)

Draw a systems diagram for the relationship between global mean
temperature, atmospheric CO<sub>2</sub> concentrations, and ocean
CO<sub>2</sub> concentrations. What are the signs of the interactions
between these components and why? What does this suggest about the
overall feedback between temperature and the ocean carbon cycle?

In [7]:

"""
                    (+)
Atmospheric CO₂ ─────────►  Global Mean Temperature
 │                              │
 │                              ▼ (-)
 ▲                              |
 │                         Ocean CO₂ 
 │                              │
 └───────────(-)───────────────-┘
"""

"                    (+)\nAtmospheric CO₂ ─────────►  Global Mean Temperature\n │                              │\n │                              ▼ (-)\n ▲                              |\n │                         Ocean CO₂ \n │                              │\n └───────────(-)───────────────-┘\n"

**Problem 1 SOLUTION:**

Henry's Law says that solubility of CO₂ in seawater decreases as temperature increases. In other words, warm water holds less CO₂. Because of this, the arrow pointing from global mean temperature to the concentration of CO₂ in the ocean is a negative (-) sign and this is a dampening relationship. As temperature increases, the ocean's water warms and CO₂ concentrations decrease. 

When concentrations of CO₂ in the ocean increase, the concentration of CO₂ in the atmosphere decrease since the ocean is acting as a carbon sink and taking out CO₂ from the atmosphere. Because of this, the sign is negative (-) and this is a dampening relationship.

Finally, when CO₂ concentrations in the atmosphere increases, the greenhouse effect is magnified since the CO₂ is now absorbing heat and re-emitting it. As a result, the global mean temperature to increase. This is an amplifying relationship and the arrow's sign is positive (+). 

This suggests that the overall feedback between temperature and the ocean carbon cycle is an amplyfing (or positive) feedback because an increase in atmospheric CO₂, increases temperature, which creates less CO2 in the ocean, which means less CO₂ is being absorbed by the ocean and there is more of it in the atmosphere, which finally leads to even more warming. 

> **Tip**
>
> Think about Henry’s law for CO<sub>2</sub>.

### Problem 2 (15 points)

A river which flows at 10 km/d is receiving discharges of wastewater
contaminated with CRUD from two sources which are 15 km apart, as shown
in the Figure below. CRUD decays exponentially in the river at a rate of
0.36 $\mathrm{d}^{-1}$ and is deposited by the atmosphere along the
river at a rate of 54 kg/km/d. 

<figure>
<img src="attachment:figures/river_diagram.png"
alt="Schematic of the river system in Problem 2" />
<figcaption aria-hidden="true">Schematic of the river system in Problem
2</figcaption>
</figure>

#### Problem 2.1

Draw a systems diagram with the relevant control volume(s) denoted and
any relevant in/out-flows between these boxes. How did you decide how
many boxes were needed?


**2.1 Solution:**

                |
                |
           -----|-----            
          |     ▼     |
         --->       ----> 
          |           |
          └───────────

Two boxes or control volumes are needed for this because there are two input streams, one before X = 15 km and one after. These streams are causing a change in flow rate of the river so the control volumes will help model and understand those changes.

#### Problem 2.2

Develop a model for the concentration of CRUD downriver by formulating
and solving the appropriate differential equation(s) analytically.

> **Tip**
>
> Formulate your model in terms of distance downriver, rather than
> leaving it in terms of time from discharge.

In [13]:

Q = 250000 # river flow rate (m^3/d)
Ci = 0.5/1000 # initial CRUD concentration in river (kg/m^3)
Q1 = 40000 # first wastewater input (kg/m^3)
C1 = 9/1000 # concentration in first wastewater input (kg/m^3)
Q2 = 60000 # second WW input (kg/m^3)
C2 = 7/1000 # concentration in second WW input (kg/m^3)

k = 0.36 # first order decay rate (1/day)
d = 54 # rate CRUD is deposited by the atmosphere along the river (kg/km/d)
v = 10 # constant average velocity of the river (km/day)
L = 15 # distance between WW input 1 and 2 (km)
limit = 2.3/1000 #regulatory CRUD limit (kg/m^3)

# diff eqn before reaching first CRUD input (x = 0)
C_upstream = (v*d)/(k*Q) # dC/dx = (-kC/v) + d/Q when dC/dx = 0 because steady state

# fcn to find the flow and concentration output of two streams mixing
function two_streams_mix(Q1, C1, Q2, C2)
    Q_out = Q1 + Q2
    C_out = ((Q1*C1) + (Q2*C2)) / Q_out
    return (Q_out, C_out)
end

# after first WW input streams
(Q3, C3) = two_streams_mix(Q, Ci, Q1, C1)


(290000, 0.0016724137931034483)


#### Problem 2.3

Determine if the system in compliance with a regulatory limit of
$2.3\ \text{kg}/(1000 \text{m}^3)$. You can do this analytically or
computationally.

### Problem 3 (10 points)

In class, we discussed the ice-albedo feedback and its possible
influence on melting the hypothesized [“Snowball
Earth”](https://en.wikipedia.org/wiki/Snowball_Earth). In this problem,
we’ll introduce a simple model of the Earth’s energy balance with
includes this feedback and examine the stability of the climate.

This simple model of the energy balance (averaged over the entire
planet) is:

<span id="eq-climate">$$
\underbrace{C\frac{dT}{dt}}_{\text{change in heat}} = \underbrace{\frac{(1-\alpha)S}{4}}_{\text{incoming radiation}} - \underbrace{(A - BT)}_{\text{outgoing radiation}} + \underbrace{a\ln \left(\frac{[CO_2]}{[CO_2]_{PI}}\right)}_{\text{greenhouse effect}},
 \qquad(1)$$</span>

where:

-   $T$ is the Earth’s global mean temperature (in $^\circ\text{C}$);
-   $C$ is the heat capacity of the atmosphere and shallow ocean, taken
    to be $51\ \text{J}/\text{m}^2/^\circ\text{C}$;
-   $\alpha$ is the planetary albedo, or the fraction of incoming
    radiation reflected by the Earth, which has a present-day value of
    approximately 0.3;
-   $S$ is the solar constant, or the amount of solar radiation received
    by the Earth averaged over area, which has a present-day value of
    $1368\ \text{W}/\text{m}^2$ but during the Neoproteorozoic Era had a
    value of $1272\ \text{W}/\text{m}^2$ (this is divided by four
    because the Earth is a sphere but $S$ is the radiation captured by a
    disc with radius equal to that of the Earth);
-   $A$ and $B$ are coefficients from the linearization of outgoing
    radiation physics, and have corresponding values
    $B=-1.3\ \text{W}/\text{m}^2/^\circ\text{C}$ (estimated from a
    number of lines of evidence about the sensitivity of outgoing
    radiation to temperature; this is negative due to a sign convention
    about the direction of incoming vs. outgoing radiation) and
    $A=221.2\ \text{W}/\text{m}^2$ (estimated by assuming the
    pre-industrial temperature of $14^\circ\text{C}$ was stable without
    anthropogenic greenhouse gas emissions).

We will ignore the greenhouse effect term as we are considering the
Earth system well before humans were around.

#### Problem 3.1

Discretize the climate model
(<a href="#eq-climate" class="quarto-xref">Equation 1</a>) using forward
Euler integration and a time step of $\Delta t = 1$ yr.

#### Problem 3.2

Rather than assuming a constant value for the albedo $\alpha$, we will
represent the ice-albedo feedback by letting $\alpha$ depend on $T$:

$$\alpha(T) = 
    \begin{cases} 
        \alpha_i & \quad \text{if } T \leq -10^\circ \text{C} \\
        \alpha_i + (\alpha_0 - \alpha_i)((T+10) / 20) & \quad \text{if } -10^\circ \text{C} \leq T \leq 10^\circ \text{C} \\
        \alpha_0 & \quad \text{if } T \geq 10^\circ \text{C}.
    \end{cases}$$

Let $\alpha_i = 0.5$ and $\alpha_0 = 0.3$. Simulate the simple climate
model using the Neoprotereozoic value of $S$ with temperature-varying
albedo for initial values of $T$ spanning
$-60^\circ\text{C} \leq T_0 \leq 30^\circ\text{C}$ over a period of 200
years. Plot the temperature trajectories. How many equilibria are there
and what are their stabilities?

#### Problem 3.3

One might hypothesize that one cause for the transition from Snowball
Earth (the stable frozen equilibrium from Problem 3.2) was an increasing
amount of incoming solar radiation (as expressed by an increase in $S$).
Examine this hypothesis using our simple model. Is this factor enough to
cause the planet to warm to the pre-industrial temperature of
$14^\circ\text{C}$?

## References

List any external references consulted, including classmates.